# Data Mining - A.A. 2025/2026
## Progetto Gruppo G3.3
### Membri:
- Masala Gabriele: 60/61/66245 (60/99/00004)
- Mantega Gabriele: 60/61/66251 (60/99/00006)
- Aresu Matteo: 60/99/00014

# 1 - Analisi del Dataset

In [1]:
# Dichiarazione Import
import pandas as pd
import numpy as np
import os
from sklearn import datasets
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction import text as sktext

from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [2]:
# Direttive di Pandas per avere un output più leggibile
pd.set_option('display.max_colwidth', 100)

# Caricamento DataFrame da sklearn
categories = ['rec.autos', 'rec.sport.baseball', 'sci.electronics', 'sci.med', 'sci.space']
train_dataset = datasets.fetch_20newsgroups(subset='train', categories=categories)
test_dataset = datasets.fetch_20newsgroups(subset='test', categories=categories)
X_train = pd.DataFrame(train_dataset.data).rename(columns={0: 'raw_text'})
y_train = pd.DataFrame(train_dataset.target).rename(columns={0: 'category'})
X_test = pd.DataFrame(test_dataset.data).rename(columns={0: 'raw_text'})
y_test = pd.DataFrame(test_dataset.target).rename(columns={0: 'category'})

# Stampo la descrizione del dataset
#print(train_dataset.DESCR[90:845])

In [3]:
# Stampo il Training Set
#display(X_train)

# Stampo il Test Set
#display(y_train)

In [4]:
# Stampo a schermo il primo record del Dataset
#print(X_train['raw_text'][0])

# 2 - Preprocessing

In [5]:
# FUNZIONI DI PREPROCESSING

# Load GloVe embeddings into a dictionary (presa da spot-intelligence)
def load_embeddings(file_path):
    embeddings = {}
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File {file_path} not found. Please check the path and try again.")
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            vector = np.asarray(values[1:], dtype='float32')
            embeddings[word] = vector
    return embeddings

# ---

# esegue: lowercase, tokenizzazione, rimozione della punteggiatura e delle stopword
def preprocess_text(text):
    # Converti il testo in minuscolo
    text = text.lower()
    
    # Rimuovi la punteggiatura
    text = ''.join(char for char in text if char.isalnum() or char.isspace())
    
    # Tokenizza il testo
    tokens = text.split()
    
    # Rimuovi le stopword (opzionale, puoi usare una lista di stopword)
    stopwords = sktext.ENGLISH_STOP_WORDS # stopwords di sklearn
    tokens = [tkn for tkn in tokens if tkn not in stopwords]
    
    return tokens

# ---

# Calcola l'embedding del documento come media degli embedding delle parole
def get_document_embedding(tokens, glove_embeddings):
    # Ottieni gli embedding per ogni parola nel documento
    word_embeddings = [glove_embeddings[token] for token in tokens if token in glove_embeddings.keys()] # Verifica se la parola ha un embedding disponibile
    
    # Calcola la media degli embedding per ottenere l'embedding del documento
    if word_embeddings:
        document_embedding = np.mean(word_embeddings, axis=0) 
    else:
        document_embedding = np.zeros(len(word_embeddings[0]))  # Se nessuna parola ha embedding, restituisci un vettore di zeri
    
    return document_embedding

# ---

# ...


## 2.1 Tf-Idf

In [6]:

# Utilizzo la classe TfidfVectorizer per convertire i record testuali in record numerici
tfidf_vectorizer = TfidfVectorizer(stop_words='english', max_features=100)
tfidf_train = tfidf_vectorizer.fit_transform(X_train['raw_text'])
tfidf_test = tfidf_vectorizer.fit_transform(X_test['raw_text'])

## 2.2 Word Embeddings

In [7]:
# caricamento degli embeddings GloVe
glove_embeddings_path = 'glove.6B.100d.txt'  
glove_embeddings = load_embeddings(glove_embeddings_path)

# Preprocessamento del testo e calcolo degli embedding dei documenti
rows_train = []
for text in X_train['raw_text']:
    tokens = preprocess_text(text)
    rows_train.append(get_document_embedding(tokens, glove_embeddings))

rows_test = []
for text in X_test['raw_text']:
    tokens = preprocess_text(text)
    rows_test.append(get_document_embedding(tokens, glove_embeddings))

# Creazione di un DataFrame per visualizzare i risultati
glove_train = np.array(rows_train)
glove_test = np.array(rows_test)

# 3. Classificazione

In [8]:
# Inizializzazione classificatori (#todo parametri sparati, dobbiamo migliorarli)
classifiers = {
    'Decision Tree': DecisionTreeClassifier(max_depth=15, min_samples_split=5, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5, weights='distance'),
    'MLP': MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42),
    'AdaBoost': AdaBoostClassifier(n_estimators=50, learning_rate=1.0, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
}

# Inizializzazione delle rappresentazioni dei dati
rappresentations = {
    'TF-IDF': (tfidf_train, tfidf_test),
    'GloVe': (glove_train, glove_test)
}


In [18]:
# Alleniamo e valutiamo tutti i modelli

results = []
for rappr, splits in rappresentations.items():
    perf = {}
    print(f"\nNow testing on {rappr} :3")
    X_train, X_test = splits
    
    for name, clf in classifiers.items():
        
        # Allenamento modello
        clf.fit(X_train, y_train.values.ravel())
        y_pred = clf.predict(X_test)
        
        perf[name] = {
        'accuracy': accuracy_score(y_test, y_pred),
            'precision': precision_score(y_test, y_pred, average='weighted', zero_division=0),
            'recall': recall_score(y_test, y_pred, average='weighted', zero_division=0),
            'f1': f1_score(y_test, y_pred, average='weighted', zero_division=0)
        }
        print(f"Training {name}...OK")
    results.append((rappr, perf))

# Mostro i risultati :3
for rappr, perf in results:
    print("\n" + "="*70)
    print(f"Classification {rappr} Summary")
    print("="*70)
    print((pd.DataFrame(perf).T).round(4))


Now testing on TF-IDF :3
Training Decision Tree...OK
Training KNN...OK
Training MLP...OK
Training Random Forest...OK
Training AdaBoost...OK
Training XGBoost...OK

Now testing on GloVe :3
Training Decision Tree...OK
Training KNN...OK
Training MLP...OK
Training Random Forest...OK
Training AdaBoost...OK
Training XGBoost...OK

Classification TF-IDF Summary
               accuracy  precision  recall      f1
Decision Tree    0.3041     0.3517  0.3041  0.3027
KNN              0.3431     0.3534  0.3431  0.3464
MLP              0.3193     0.3523  0.3193  0.3218
Random Forest    0.3335     0.3845  0.3335  0.3228
AdaBoost         0.3305     0.3930  0.3305  0.3200
XGBoost          0.2945     0.3653  0.2945  0.2899

Classification GloVe Summary
               accuracy  precision  recall      f1
Decision Tree    0.6594     0.6594  0.6594  0.6589
KNN              0.8765     0.8780  0.8765  0.8770
MLP              0.8882     0.8882  0.8882  0.8880
Random Forest    0.8725     0.8736  0.8725  0.8728
Ad